# Credit Card Approval Prediction

> **Academic group project** completed as part of a college academic internship. This repository documents collaborative coursework; individual contributions should be credited separately.

## Before running
1. Obtain the project dataset (`credit card_dataset.csv`) from the original authorised course/internship source.
2. Place the CSV in the same directory as this notebook.
3. Install the dependencies listed in `requirements.txt`.
4. Run the notebook from top to bottom.

The dataset is intentionally not bundled here because its redistribution permissions have not been verified. Do not publish restricted or personally identifiable data.



In [ ]:
import numpy as np
import pandas as pd

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from scipy.stats import chi2_contingency
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [ ]:
import math

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
file_path="credit card_dataset.csv"

In [ ]:
cdf = pd.read_csv("credit card_dataset.csv")

In [ ]:
cdf.shape

In [ ]:
cdf.head()

In [ ]:
cdf.info()

In [ ]:
print(cdf.isnull().sum())

In [ ]:
cdf.duplicated().sum()

In [ ]:
cdf.drop(columns=['ZipCode'], inplace=True)

In [ ]:
eda_df = cdf.copy()

In [ ]:
cdf1=cdf.copy()

In [ ]:
industry_mapping = {
    'Energy': 'Energy & Utilities',
    'Materials': 'Energy & Utilities',
    'Utilities': 'Energy & Utilities',
    'ConsumerDiscretionary': 'Consumer & Retail',
    'ConsumerStaples': 'Consumer & Retail',
    'Healthcare': 'Public & Healthcare',
    'Education': 'Public & Healthcare',
    'Research': 'Public & Healthcare',
    'Financials': 'Finance & Real Estate',
    'Real Estate': 'Finance & Real Estate',
    'InformationTechnology': 'Tech & Communication',
    'CommunicationServices': 'Tech & Communication',
    'Industrials': 'Industrial & Transport',
    'Transport': 'Industrial & Transport'
}

In [ ]:
cdf1['Industry_Grouped'] = cdf1['Industry'].map(industry_mapping)

In [ ]:
cdf1 = pd.get_dummies(cdf1, columns=['Industry_Grouped','Ethnicity','Citizen'	], drop_first=True, dtype=int)

In [ ]:
cdf1.drop(columns=['Industry'], inplace=True)

In [ ]:
cdf1.shape

In [ ]:
cdf1.head().transpose()

In [ ]:
continuous_cols = ['Age', 'Debt', 'YearsEmployed', 'Income']
discrete_cols = ['CreditScore']
binary_cols = ['Gender', 'Married', 'BankCustomer', 'PriorDefault', 'Employed', 'DriversLicense', 'Approved']
categorical_cols = ['Industry', 'Ethnicity', 'Citizen']

In [ ]:
for col in categorical_cols:
   print(f"\nFeature: '{col}' ({eda_df[col].nunique()} unique categories)")
   freq_df = pd.DataFrame(
        {
            "Count": eda_df[col].value_counts(),
            "Percentage (%)": (eda_df[col].value_counts(normalize=True) * 100).round(2)
        }
    )
   print(freq_df)

In [ ]:
fig, axes = plt.subplots(1, len(categorical_cols), figsize=(18, 5))

for idx, col in enumerate(categorical_cols):
  sns.countplot(
      data=eda_df,
      y=col,
      ax=axes[idx],
      order=eda_df[col].value_counts().index,
      palette='viridis',
      hue=col,
      legend=False,
  )
  axes[idx].set_title(f'Count of {col} ', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
for col in binary_cols:
    prop_1 = eda_df[col].value_counts(normalize=True).get(1,0)*100
    counts = eda_df[col].value_counts().to_dict()
    print(
        f"Column '{col}': {counts} | {prop_1:.1f}% positive flags (1s)"
    )

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for idx, col in enumerate(binary_cols):
  sns.countplot(
      data=eda_df,
      x=col,
      ax=axes[idx],
      order=eda_df[col].value_counts().index,
      palette='mako'
  )
  axes[idx].set_title(f'Count of {col} (Binary)')

for idx in range(len(binary_cols), len(axes)):
    fig.delaxes(axes[idx])
plt.tight_layout()
plt.show()

In [ ]:
num_summary = pd.concat([
    eda_df[continuous_cols].describe().round(2).T,
    eda_df[continuous_cols].skew().round(2).rename('skewness'),
    eda_df[continuous_cols].kurtosis().round(2).rename('kurtosis')
], axis=1)
display(num_summary)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for idx, col in enumerate(continuous_cols):
    sns.histplot(data=eda_df, x=col, kde=True, ax=axes[idx], color='royalblue', bins=30)
    axes[idx].set_title(f'Distribution of Continuous: {col}')
    axes[idx].set_xlabel('')

plt.tight_layout()
plt.show()

In [ ]:
print("CreditScore Summary (Quantitative) ")
print(eda_df['CreditScore'].describe())


In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(
    data=eda_df,
    x='CreditScore',
    discrete=True,
    color='darkorange',
    kde=True
)
plt.title('Credit Score Distribution (Discrete Histogram)')
plt.xlabel('Credit Score')
plt.ylabel('Count')
plt.show()

In [ ]:
num_cols = continuous_cols + discrete_cols

In [ ]:
corr_matrix = eda_df[num_cols].corr()
print(corr_matrix)

In [ ]:
plt.figure(figsize=(8, 6))
corr = eda_df[num_cols].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap of Numerical Features')
plt.tight_layout()
plt.show()

In [ ]:
df_vif = eda_df[num_cols].dropna()
vif_data = pd.DataFrame()
vif_data['Feature'] = df_vif.columns
vif_data['VIF'] = [variance_inflation_factor(df_vif.values, i) for i in range(len(df_vif.columns))]
print("\nVariance Inflation Factor (VIF):\n", vif_data)

In [ ]:
target = 'Approved'

In [ ]:
chi2_results = []
for col in binary_cols:
    contingency_table = pd.crosstab(eda_df[col], eda_df[target])
    chi2, p, dof, expected = chi2_contingency(contingency_table)
    chi2_results.append({'Feature': col, 'Chi2': chi2, 'p-value': p})

chi2_df = pd.DataFrame(chi2_results)
print("\nChi-Square Test Results:\n", chi2_df)

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, col in enumerate(num_cols):
    sns.histplot(
        data=eda_df,
        x=col,
        hue=target,
        kde=True,
        ax=axes[idx],
        common_norm=False,
        palette='Set2'
    )
    axes[idx].set_title(f'Distribution of {col} by {target}')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Density / Count')

axes[-1].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
plot_cats = categorical_cols + [col for col in binary_cols if col != target]

# Dynamically calculate rows needed for 2 columns per row
n_cols = 2
n_rows = math.ceil(len(plot_cats) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 5 * n_rows))
axes = axes.flatten()

for idx, col in enumerate(plot_cats):
    sns.countplot(data=eda_df, x=col, hue=target, ax=axes[idx], palette='Set1')
    axes[idx].set_title(f'{col} vs {target}')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Count')
    axes[idx].tick_params(axis='x', rotation=45)

# Hide any empty unused subplots in the grid
for idx in range(len(plot_cats), len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,classification_report, confusion_matrix)

In [ ]:
X = cdf1.drop(columns=['Approved'])
y = cdf1['Approved']

In [ ]:
test_sizes = [0.2, 0.25, 0.3, 0.35, 0.4]

1. Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

for ts in test_sizes:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=ts, random_state=42, stratify=y)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    print(f"~ Logistic Regression | Test size = {ts} ~")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("Classification Report:\n", classification_report(y_test, y_pred))
    print()

2. KNN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

for ts in test_sizes:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=ts, random_state=42, stratify=y)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = KNeighborsClassifier()
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    print(f"~ KNN | Test size = {ts} ~")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("Classification Report:\n", classification_report(y_test, y_pred))
    print()

3. Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

for ts in test_sizes:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=ts, random_state=42, stratify=y)
    model = DecisionTreeClassifier(random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print(f"~ Decision Tree | Test size = {ts} ~")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("Classification Report:\n", classification_report(y_test, y_pred))
    print()

4. Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

for ts in test_sizes:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=ts, random_state=42, stratify=y)
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print(f"~ Random Forest | Test size = {ts} ~")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("Classification Report:\n", classification_report(y_test, y_pred))
    print()

5. XGBoost

In [ ]:
from xgboost import XGBClassifier

for ts in test_sizes:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=ts, random_state=42, stratify=y)
    model = XGBClassifier(random_state=42, eval_metric='logloss')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print(f"~ XGBoost | Test size = {ts} ~")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("Classification Report:\n", classification_report(y_test, y_pred))
    print()

6. SVM

In [ ]:
from sklearn.svm import SVC

for ts in test_sizes:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=ts, random_state=42, stratify=y)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = SVC(random_state=42)
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    print(f"~ SVM | Test size = {ts} ~")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("Classification Report:\n", classification_report(y_test, y_pred))
    print()

7. Naive Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB

for ts in test_sizes:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=ts, random_state=42, stratify=y)
    model = GaussianNB()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print(f"~ Naive Bayes | Test size = {ts} ~")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("Classification Report:\n", classification_report(y_test, y_pred))
    print()

# **Cross-Validation**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
# Logistic Regression CV
model = LogisticRegression(random_state=42, max_iter=1000)
cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='accuracy')
print("Logistic Regression CV Accuracy:", cv_scores.mean(), "+/-", cv_scores.std())

In [ ]:
# KNN CV
model = KNeighborsClassifier()
cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='accuracy')
print("KNN CV Accuracy:", cv_scores.mean(), "+/-", cv_scores.std())

In [ ]:
# Decision Tree CV
model = DecisionTreeClassifier(random_state=42)
cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
print("Decision Tree CV Accuracy:", cv_scores.mean(), "+/-", cv_scores.std())

In [ ]:
# Random Forest CV
model = RandomForestClassifier(n_estimators=100, random_state=42)
cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
print("Random Forest CV Accuracy:", cv_scores.mean(), "+/-", cv_scores.std())

In [ ]:
# XGBoost CV
model = XGBClassifier(random_state=42, eval_metric='logloss')
cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
print("XGBoost CV Accuracy:", cv_scores.mean(), "+/-", cv_scores.std())

In [ ]:
# SVM CV
model = SVC(random_state=42)
cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='accuracy')
print("SVM CV Accuracy:", cv_scores.mean(), "+/-", cv_scores.std())

In [ ]:
# Naive Bayes CV
model = GaussianNB()
cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
print("Naive Bayes CV Accuracy:", cv_scores.mean(), "+/-", cv_scores.std())

# **Fine-Tuning**

In [ ]:
# Logistic Regression Tuning
param_grid_lr = {"C": [0.01, 0.1, 1, 10, 100]}
grid_lr = GridSearchCV(LogisticRegression(random_state=42, max_iter=1000), param_grid_lr, cv=cv, scoring='accuracy', n_jobs=-1)
grid_lr.fit(X_train_scaled, y_train)
print("Best params:", grid_lr.best_params_)
print("Best CV Accuracy:", grid_lr.best_score_)

best_lr = grid_lr.best_estimator_
y_pred = best_lr.predict(X_test_scaled)
print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("Test F1:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

In [ ]:
# KNN Tuning
param_grid_knn = {"n_neighbors": [3, 5, 7, 9, 11, 15], "weights": ["uniform", "distance"]}
grid_knn = GridSearchCV(KNeighborsClassifier(), param_grid_knn, cv=cv, scoring='accuracy', n_jobs=-1)
grid_knn.fit(X_train_scaled, y_train)
print("Best params:", grid_knn.best_params_)
print("Best CV Accuracy:", grid_knn.best_score_)

best_knn = grid_knn.best_estimator_
y_pred = best_knn.predict(X_test_scaled)
print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("Test F1:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

In [ ]:
# Decision Tree Tuning
param_grid_dt = {"max_depth": [None, 3, 5, 7, 10], "min_samples_split": [2, 5, 10], "min_samples_leaf": [1, 2, 4]}
grid_dt = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid_dt, cv=cv, scoring='accuracy', n_jobs=-1)
grid_dt.fit(X_train, y_train)
print("Best params:", grid_dt.best_params_)
print("Best CV Accuracy:", grid_dt.best_score_)

best_dt = grid_dt.best_estimator_
y_pred = best_dt.predict(X_test)
print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("Test F1:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

In [ ]:
# Random Forest Tuning
param_grid_rf = {"n_estimators": [100, 200, 300], "max_depth": [None, 5, 10, 15], "min_samples_split": [2, 5, 10], "min_samples_leaf": [1, 2, 4]}
grid_rf = GridSearchCV(RandomForestClassifier(random_state=42), param_grid_rf, cv=cv, scoring='accuracy', n_jobs=-1)
grid_rf.fit(X_train, y_train)
print("Best params:", grid_rf.best_params_)
print("Best CV Accuracy:", grid_rf.best_score_)

best_rf = grid_rf.best_estimator_
y_pred = best_rf.predict(X_test)
print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("Test F1:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

In [ ]:
# XGBoost Tuning
param_grid_xgb = {"n_estimators": [100, 200, 300], "max_depth": [3, 5, 7], "learning_rate": [0.01, 0.05, 0.1, 0.2], "subsample": [0.8, 1.0]}
grid_xgb = GridSearchCV(XGBClassifier(random_state=42, eval_metric='logloss'), param_grid_xgb, cv=cv, scoring='accuracy', n_jobs=-1)
grid_xgb.fit(X_train, y_train)
print("Best params:", grid_xgb.best_params_)
print("Best CV Accuracy:", grid_xgb.best_score_)

best_xgb = grid_xgb.best_estimator_
y_pred = best_xgb.predict(X_test)
print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("Test F1:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

In [ ]:
# SVM Tuning
param_grid_svc = {"C": [0.1, 1, 10, 100], "kernel": ["linear", "rbf"], "gamma": ["scale", "auto"]}
grid_svc = GridSearchCV(SVC(random_state=42), param_grid_svc, cv=cv, scoring='accuracy', n_jobs=-1)
grid_svc.fit(X_train_scaled, y_train)
print("Best params:", grid_svc.best_params_)
print("Best CV Accuracy:", grid_svc.best_score_)

best_svc = grid_svc.best_estimator_
y_pred = best_svc.predict(X_test_scaled)
print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("Test F1:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

In [ ]:
# Naive Bayes Tuning
param_grid_nb = {"var_smoothing": np.logspace(-11, -7, 20)}
grid_nb = GridSearchCV(GaussianNB(), param_grid_nb, cv=cv, scoring='accuracy', n_jobs=-1)
grid_nb.fit(X_train, y_train)
print("Best params:", grid_nb.best_params_)
print("Best CV Accuracy:", grid_nb.best_score_)

best_nb = grid_nb.best_estimator_
y_pred = best_nb.predict(X_test)
print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("Test F1:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

# **ANN**

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from itertools import product

In [ ]:
histories = {}
test_size_results = {}

for ts in test_sizes:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=ts, random_state=42, stratify=y)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = Sequential()
    model.add(Dense(units=16, activation='relu', input_dim=X_train_scaled.shape[1]))
    model.add(Dropout(0.2))
    model.add(Dense(units=8, activation='relu'))
    model.add(Dense(units=1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    history = model.fit(X_train_scaled, y_train, epochs=50, batch_size=32,
                         validation_split=0.2, verbose=0)

    loss, accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)
    y_pred_prob = model.predict(X_test_scaled, verbose=0)
    y_pred = (y_pred_prob > 0.5).astype(int)

    histories[ts] = history
    test_size_results[ts] = {"accuracy": accuracy, "f1": f1_score(y_test, y_pred)}

    print(f"~ ANN (TensorFlow) | Test size = {ts} ~")
    print(f"Test Accuracy: {accuracy:.4f}")
    print("F1 Score:", f1_score(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("Classification Report:\n", classification_report(y_test, y_pred))
    print()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, ts in enumerate(test_sizes):
    axes[i].plot(histories[ts].history['accuracy'], label='Train')
    axes[i].plot(histories[ts].history['val_accuracy'], label='Validation')
    axes[i].set_title(f"Test size = {ts}")
    axes[i].set_xlabel("Epoch")
    axes[i].set_ylabel("Accuracy")
    axes[i].legend()

axes[-1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(test_sizes, [test_size_results[ts]["accuracy"] for ts in test_sizes], marker='o', label='Accuracy')
plt.plot(test_sizes, [test_size_results[ts]["f1"] for ts in test_sizes], marker='s', label='F1 Score')
plt.xlabel("Test Size")
plt.ylabel("Score")
plt.title("ANN Performance Across Test Sizes")
plt.legend()
plt.show()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_arr = np.array(X_train_scaled)
y_train_arr = np.array(y_train)

fold_scores = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X_train_arr, y_train_arr)):
    X_tr, X_val = X_train_arr[train_idx], X_train_arr[val_idx]
    y_tr, y_val = y_train_arr[train_idx], y_train_arr[val_idx]

    model = Sequential()
    model.add(Dense(units=16, activation='relu', input_dim=X_tr.shape[1]))
    model.add(Dropout(0.2))
    model.add(Dense(units=8, activation='relu'))
    model.add(Dense(units=1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    model.fit(X_tr, y_tr, epochs=50, batch_size=32, verbose=0)

    val_loss, val_accuracy = model.evaluate(X_val, y_val, verbose=0)
    fold_scores.append(val_accuracy)
    print(f"Fold {fold+1} Accuracy: {val_accuracy:.4f}")

fold_scores = np.array(fold_scores)
print("\nANN CV Accuracy:", fold_scores.mean(), "+/-", fold_scores.std())

In [ ]:
param_grid_ann = {
    "units_1": [16, 32, 64],
    "units_2": [8, 16, 32],
    "dropout": [0.2, 0.3],
}

keys = list(param_grid_ann.keys())
combinations = list(product(*param_grid_ann.values()))

best_score = -np.inf
best_params = None

for combo in combinations:
    params = dict(zip(keys, combo))
    fold_scores = []

    for train_idx, val_idx in cv.split(X_train_arr, y_train_arr):
        X_tr, X_val = X_train_arr[train_idx], X_train_arr[val_idx]
        y_tr, y_val = y_train_arr[train_idx], y_train_arr[val_idx]

        model = Sequential()
        model.add(Dense(units=params["units_1"], activation='relu', input_dim=X_tr.shape[1]))
        model.add(Dropout(params["dropout"]))
        model.add(Dense(units=params["units_2"], activation='relu'))
        model.add(Dense(units=1, activation='sigmoid'))
        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
        model.fit(X_tr, y_tr, epochs=50, batch_size=32, verbose=0)

        val_loss, val_accuracy = model.evaluate(X_val, y_val, verbose=0)
        fold_scores.append(val_accuracy)

    mean_score = np.mean(fold_scores)
    print(f"Params: {params} -> CV Accuracy: {mean_score:.4f}")

    if mean_score > best_score:
        best_score = mean_score
        best_params = params

print("\nBest params:", best_params)
print("Best CV Accuracy:", best_score)

best_model = Sequential()
best_model.add(Dense(units=best_params["units_1"], activation='relu', input_dim=X_train_arr.shape[1]))
best_model.add(Dropout(best_params["dropout"]))
best_model.add(Dense(units=best_params["units_2"], activation='relu'))
best_model.add(Dense(units=1, activation='sigmoid'))
best_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
best_model.fit(X_train_arr, y_train_arr, epochs=50, batch_size=32, verbose=0)

test_loss, test_accuracy = best_model.evaluate(X_test_scaled, y_test, verbose=0)
y_pred = (best_model.predict(X_test_scaled, verbose=0) > 0.5).astype(int)

print("\nTest Accuracy:", test_accuracy)
print("Test F1:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

In [ ]:
import seaborn as sns

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("ANN Confusion Matrix (Tuned)")
plt.show()